In [5]:
# 1. Initial Setup (RUN THIS FIRST!)
import os
import cv2
import numpy as np
import pydicom
import torch
import torch.nn as nn
from tqdm import tqdm
from sklearn.model_selection import train_test_split


###os: To create folders and handle file paths.

cv2 (OpenCV): For image processing (DICOM→PNG conversion, masking).

numpy: For numerical operations on image arrays.

pydicom: To read medical DICOM files.

torch: For building/training the U-Net model (deep learning).

tqdm: To show progress bars during long operations.

train_test_split: For splitting data into training/validation sets (though we didn’t use it yet).

In [7]:
# Create folders (if they don't exist)
os.makedirs("converted_pngs", exist_ok=True)
os.makedirs("masks", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("predict_porosity", exist_ok=True)

print("✅ Folders ready: converted_pngs/, masks/, outputs/, predict_porosity/")

✅ Folders ready: converted_pngs/, masks/, outputs/, predict_porosity/


### Step 2: DICOM to PNG Converter
Goal: Convert medical-grade DICOM files (from CT scans) into standard PNG images for easier processing.

In [3]:
# 2. DICOM to PNG Converter
def convert_dicom_to_png(dicom_folder="12B_XCT", output_folder="converted_pngs"):
    dcm_files = sorted([f for f in os.listdir(dicom_folder) if f.endswith('.dcm')])
    
    for filename in tqdm(dcm_files, desc="Converting DICOMs"):
        try:
            img = pydicom.dcmread(os.path.join(dicom_folder, filename)).pixel_array
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
            cv2.imwrite(os.path.join(output_folder, filename.replace('.dcm','.png')), img)
        except Exception as e:
            print(f"❌ Error with {filename}: {str(e)}")
    
    print(f"✅ Saved {len(os.listdir(output_folder))} PNGs to '{output_folder}/'")

# Run it
convert_dicom_to_png()

Converting DICOMs: 100%|██████████| 180/180 [00:07<00:00, 24.61it/s]

✅ Saved 180 PNGs to 'converted_pngs/'


###Breakdown

1)dcm_files = sorted([...])
-->Lists all .dcm files in 12B_XCT/ and sorts them alphabetically/numerically.
-->Why? Ensures CT slices are processed in order (critical for 3D scans).

2)pydicom.dcmread(...).pixel_array
-->Reads the raw image data from DICOM files (which store metadata + pixel values).
-->Why? DICOMs aren’t directly usable by OpenCV/torch—we need to extract pixel data.

3)cv2.normalize(img, ..., cv2.NORM_MINMAX)
-->Scales pixel values to 0–255 (standard 8-bit PNG range).
-->Why? DICOMs often use 12/16-bit depth (0–4095 or 0–65535). Normalization ensures consistency.

4).astype('uint8')
-->Converts the image to 8-bit unsigned integers (PNG format).
-->Why? PNGs don’t support higher bit depths by default.

5)cv2.imwrite(...)
-->Saves the normalized image as a PNG in converted_pngs/.

6)Error Handling (try/except)
-->Skips corrupt DICOM files and logs errors.

### Step 3: Mask Generator
Goal: Create binary masks from PNGs (white = pores, black = rock) to train the U-Net model.

In [6]:
# 3. Mask Generator
def create_masks(image_folder="converted_pngs", output_folder="masks"):
    png_files = sorted([f for f in os.listdir(image_folder) if f.endswith('.png')])
    
    for filename in tqdm(png_files, desc="Creating masks"):
        img = cv2.imread(os.path.join(image_folder, filename), cv2.IMREAD_GRAYSCALE)
        # Automatic thresholding
        _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        # Remove noise
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
        cv2.imwrite(os.path.join(output_folder, filename), mask)
    
    print(f"✅ Saved {len(os.listdir(output_folder))} masks to '{output_folder}/'")

# Run it
create_masks()

Creating masks: 100%|██████████| 180/180 [00:07<00:00, 23.01it/s]


✅ Saved 180 masks to 'masks/'


###Breakdown

1)png_files = sorted([...])
-->Lists all .png files (from Step 2) and sorts them.
-->Why? Ensures masks align perfectly with the original images.

2)cv2.imread(..., cv2.IMREAD_GRAYSCALE)
-->Loads PNGs as grayscale (1 channel).
-->Why? Color isn’t needed—pores and rock differ in brightness, not color.

3)cv2.threshold(..., THRESH_BINARY_INV + THRESH_OTSU)
-->Otsu’s Method: Automatically calculates the optimal threshold to separate pores (dark) from rock (bright).
-->BINARY_INV: Inverts the result so pores become white (255) and rock black (0).
-->Why? Manual thresholds fail across diverse images—Otsu adapts to each image.

4)cv2.morphologyEx(..., MORPH_OPEN)
-->Applies morphological opening (erosion followed by dilation) with a 3x3 kernel.
-->Why? Removes tiny noise spots (salt-and-pepper) while preserving pore shapes.

5)cv2.imwrite(...)
-->Saves the cleaned mask to masks/.

In [ ]:
Impact
Before: Raw PNGs with grayscale values.

After: Noise-free binary masks for training U-Net.

### Step 4: U-Net Model Definition
Goal: Build the neural network architecture that will learn to segment pores from rock images.

In [9]:
# 4. U-Net Model Definition
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU()
            )
        
        # Encoder
        self.enc1 = conv_block(1, 64)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = conv_block(256, 512)
        
        # Decoder
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)
        
        self.final = nn.Conv2d(64, 1, 1)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e3))
        
        # Decoder
        d3 = self.dec3(torch.cat([e3, self.up3(b)], dim=1))
        d2 = self.dec2(torch.cat([e2, self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([e1, self.up1(d2)], dim=1))
        
        return torch.sigmoid(self.final(d1))

print("✅ U-Net model defined!")

✅ U-Net model defined!


###Why U-Net?
Medical Imaging Standard: Originally for biomedical segmentation.

Handles Small Datasets: Skip connections prevent overfitting.

Precise Boundaries: Recovers spatial details via upsampling.

### Step 5: Training Setup
Goal: Train the U-Net to predict pore masks from rock images using the prepared dataset.


In [ ]:
# 5. Training Setup (FIXED VERSION)
import torch
from torch.utils.data import Dataset, DataLoader  # <-- THIS WAS MISSING!
from tqdm import tqdm

# Define the Dataset class FIRST
class RockDataset(Dataset):
    def __init__(self, image_dir="converted_pngs", mask_dir="masks"):
        self.files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
        self.image_dir = image_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = cv2.imread(os.path.join(self.image_dir, self.files[idx]), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(os.path.join(self.mask_dir, self.files[idx]), cv2.IMREAD_GRAYSCALE)
        return (
            torch.tensor(img/255.).unsqueeze(0).float(),  # Image (1, 128, 128)
            torch.tensor(mask/255.).unsqueeze(0).float()  # Mask (1, 128, 128)
        )
#-->Why?
#Loads image-mask pairs from disk
#Normalizes pixel values to [0,1] (required for neural networks)
#Adds channel dimension (1 for grayscale)

# Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

# Create DataLoader
dataset = RockDataset()  # <-- Now this will work!
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

#Key Choices:
#Adam optimizer: Adaptive learning rate for stable training
#BCELoss: Standard for binary segmentation
#Batch size 4: Balances memory usage and gradient stability

# Training Loop
for epoch in range(5):  # Quick test with 5 epochs
    model.train()
    epoch_loss = 0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/5"):
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.4f}")

# Save model
torch.save(model.state_dict(), "unet_rock_model.pth")
print("✅ Training complete! Model saved as 'unet_rock_model.pth'")

### Step 6: Porosity Prediction & Analysis
Goal: Use the trained U-Net to predict pore structures and calculate porosity percentages.

In [ ]:
import os
import cv2
import numpy as np
import torch
from tqdm import tqdm

def predict_porosity_accurate(model_path="unet_rock_model.pth", 
                            input_dir="predict_porosity",
                            output_dir="outputs",
                            target_size=(128, 128),
                            min_pore_size=5):
    """
    Final professional-grade porosity prediction with:
    - Advanced preprocessing
    - Size-based pore filtering
    - Confidence thresholding
    - Multi-step validation
    """
    
    # Load model with verification
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file {model_path} not found")
        
    model = UNet()
    try:
        model.load_state_dict(torch.load(model_path, map_location="cpu"))
    except:
        raise ValueError("Failed to load model - check architecture compatibility")
    model.eval()
    
    # Get validated image list
    image_files = []
    for f in os.listdir(input_dir):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            full_path = os.path.join(input_dir, f)
            if os.path.getsize(full_path) > 0:  # Check not empty
                image_files.append(f)
    
    if not image_files:
        print("⚠️ No valid images found in", input_dir)
        return []

    # Processing pipeline
    results = []
    for filename in tqdm(image_files, desc="Professional Analysis"):
        try:
            # 1. Load with verification
            img_path = os.path.join(input_dir, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"⚠️ Corrupted image: {filename}")
                continue
                
            original_shape = img.shape
            
            # 2. Advanced normalization with CLAHE
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            img = clahe.apply(img)
            
            # 3. Intelligent inversion
            if cv2.mean(img)[0] > 127:
                img = 255 - img
                
            # 4. Artifact removal (dynamic cropping)
            border_size = int(min(img.shape) * 0.1) if min(img.shape) > 300 else 0
            img = img[border_size:-border_size, border_size:-border_size] if border_size else img
            
            # 5. Prepare for model
            img = cv2.resize(img, target_size)
            img_tensor = torch.tensor(img/255., dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            
            # 6. Prediction with confidence threshold
            with torch.no_grad():
                mask = model(img_tensor).squeeze().numpy()
            
            # 7. Dynamic thresholding with confidence cutoff
            mask_8bit = (mask * 255).astype(np.uint8)
            _, thresh = cv2.threshold(mask_8bit, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            
            # 8. Size-based pore filtering (critical fix)
            n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(thresh, 4)
            filtered_mask = np.zeros_like(thresh)
            
            for i in range(1, n_labels):
                if stats[i, cv2.CC_STAT_AREA] >= min_pore_size:
                    filtered_mask[labels == i] = 255
                    
            # 9. Edge artifact removal
            filtered_mask[:5,:] = 0
            filtered_mask[-5:,:] = 0
            filtered_mask[:,:5] = 0
            filtered_mask[:,-5:] = 0
            
            # 10. Calculate actual porosity (excluding edge artifacts)
            porosity = np.sum(filtered_mask > 0) / (filtered_mask.size - 4*(filtered_mask.shape[0]+filtered_mask.shape[1])) * 100
            
            # 11. Quality control checks
            if porosity > 50:  # Unrealistically high
                print(f"⚠️ High porosity ({porosity:.1f}%) detected in {filename} - verifying...")
                # Fallback to adaptive threshold if Otsu failed
                adaptive_thresh = cv2.adaptiveThreshold(img, 255, 
                                                      cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                                      cv2.THRESH_BINARY_INV, 11, 2)
                adaptive_porosity = np.mean(adaptive_thresh > 0) * 100
                
                # Use the more conservative estimate
                porosity = min(porosity, adaptive_porosity)
            
            # 12. Save results with metadata
            os.makedirs(output_dir, exist_ok=True)
            
            # Save detailed diagnostic image
            debug_img = np.hstack([
                cv2.resize(img, (256, 256)),
                cv2.resize(mask_8bit, (256, 256)),
                cv2.resize(thresh, (256, 256)),
                cv2.resize(filtered_mask, (256, 256))
            ])
            
            cv2.imwrite(os.path.join(output_dir, f"diagnostic_{filename}"), debug_img)
            cv2.imwrite(os.path.join(output_dir, f"final_mask_{filename}"), filtered_mask)
            
            results.append({
                'filename': filename,
                'porosity': float(porosity),
                'original_shape': original_shape,
                'processed_shape': img.shape
            })
            
            print(f"\n🔬 {filename}: {porosity:.2f}% porosity | Pores: {n_labels-1}")

        except Exception as e:
            print(f"\n❌ Professional analysis failed for {filename}: {str(e)}")
            continue

    # Generate comprehensive report
    print("\n=== PROFESSIONAL POROSITY REPORT ===")
    print(f"Analyzed {len(results)} samples")
    
    if results:
        avg_porosity = np.mean([r['porosity'] for r in results])
        median_porosity = np.median([r['porosity'] for r in results])
        
        print(f"\nAverage porosity: {avg_porosity:.2f}%")
        print(f"Median porosity: {median_porosity:.2f}%")
        print(f"Range: {min([r['porosity'] for r in results]):.2f}%-{max([r['porosity'] for r in results]):.2f}%")
        
        # Save CSV report
        import csv
        with open(os.path.join(output_dir, 'porosity_report.csv'), 'w') as f:
            writer = csv.DictWriter(f, fieldnames=results[0].keys())
            writer.writeheader()
            writer.writerows(results)
        
        print(f"\n✅ Saved full results to {output_dir}/")

    return results

# Execute with professional parameters
predict_porosity_accurate(
    min_pore_size=10,  # Minimum pore area in pixels to consider
)